# F1 full-run candidate (prepared, not launched)

Fresh `yolo11s.pt`, E1 recipe, 50% positives + 7,259 Pyro-SDIS positives + 1,327 hard negatives,
30 epochs, recall@FPR<=2% checkpoint selection. Event cameras brison-110 / courmettes-212 /
brison-200 are excluded from training. No SEALED read. Do not launch without approval.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()

In [ ]:
import importlib.metadata as md, os
os.environ["YOLO_AUTOINSTALL"]="False"; os.environ["YOLO_VERBOSE"]="False"; os.environ["YOLO_OFFLINE"]="True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
def ver(n):
    try: return md.version(n)
    except Exception: return None
if ver("ultralytics") != "8.4.147":
    get_ipython().system("pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop")
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")})

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent
CODE = sorted(Path("/kaggle/input").rglob("stage_b_train.py"))[0].parent
PYRO = sorted(Path("/kaggle/input").rglob("firesight-f1-pyro"))[0]
NEG = sorted(Path("/kaggle/input").rglob("e1_negatives.txt"))[0].parent.parent
EVALNEG = sorted(Path("/kaggle/input").rglob("eval_neg.txt"))[0].parent.parent
YOLO = sorted(Path(CODE).glob("yolo11s.pt"))[0]
print("DATA",DATA); print("CODE",CODE); print("PYRO",PYRO); print("NEG",NEG); print("EVALNEG",EVALNEG); print("YOLO",YOLO)

In [ ]:
from pathlib import Path
def resolve(lines, root):
    out=[]
    for l in lines:
        p=Path(l.strip())
        out.append(p if p.is_absolute() else (Path(root)/p).resolve())
    return out
# base 50% positives only
base=[]
for l in open(DATA/"lists/train_50.txt",encoding="utf-8"):
    if not l.strip(): continue
    p=Path(l.strip()); p=p if p.is_absolute() else (DATA/p).resolve()
    lbl=DATA/"labels"/(p.stem+".txt")
    if lbl.exists() and lbl.stat().st_size>0: base.append(p)
pyro=resolve([l for l in open(PYRO/"lists/train.txt",encoding="utf-8") if l.strip()],PYRO)
neg=resolve([l for l in open(NEG/"lists/e1_negatives.txt",encoding="utf-8") if l.strip()],NEG)
val=resolve([l for l in open(DATA/"lists/val_early.txt",encoding="utf-8") if l.strip()],DATA)
cal=resolve([l for l in open(DATA/"lists/val_cal.txt",encoding="utf-8") if l.strip()],DATA)
rep=resolve([l for l in open(DATA/"lists/val_report.txt",encoding="utf-8") if l.strip()],DATA)
indep=resolve([l for l in open(EVALNEG/"lists/eval_neg.txt",encoding="utf-8") if l.strip()],EVALNEG)
out=Path("/kaggle/working/lists"); out.mkdir(parents=True,exist_ok=True)
(out/"train.txt").write_text("\n".join(str(p) for p in base+pyro+neg)+"\n",encoding="utf-8")
(out/"val.txt").write_text("\n".join(str(p) for p in val)+"\n",encoding="utf-8")
(out/"val_cal.txt").write_text("\n".join(str(p) for p in cal)+"\n",encoding="utf-8")
(out/"val_report.txt").write_text("\n".join(str(p) for p in rep)+"\n",encoding="utf-8")
(out/"eval_neg.txt").write_text("\n".join(str(p) for p in indep)+"\n",encoding="utf-8")
print("base positives",len(base),"pyro",len(pyro),"neg",len(neg),"total",len(base)+len(pyro)+len(neg))
assert len(base)==13063 and len(pyro)==7259 and len(neg)==1327

In [ ]:
import subprocess
cmd=["python","-u",str(Path(CODE)/"stage_b_train.py"),"--dataset",str(DATA),
     "--train-list","/kaggle/working/lists/train.txt","--val-list","/kaggle/working/lists/val.txt",
     "--val-cal-list","/kaggle/working/lists/val_cal.txt","--out","/kaggle/working/runs/F1",
     "--model",str(YOLO),"--epochs","30","--patience","10","--imgsz","640","--batch","64",
     "--device","0","--workers","4","--lr0","0.01","--lrf","0.01","--seed","0","--amp"]
print("RUN"," ".join(cmd),flush=True); subprocess.run(cmd,check=True)

In [ ]:
import subprocess, json
# fixed eval: chunked prediction, no full-list OOM
for name,lst in (("report","/kaggle/working/lists/val_report.txt"),("indep","/kaggle/working/lists/eval_neg.txt")):
    subprocess.run(["python",str(Path(CODE)/"stage_b_predict.py"),"--model","/kaggle/working/runs/F1/weights/best_recall.pt",
                    "--list",lst,"--out",f"/kaggle/working/preds_F1_{name}.jsonl","--chunk","8","--batch","2",
                    "--device","0"],check=True)
for t in ("0.009","0.015","0.02","0.03"):
    subprocess.run(["python",str(Path(CODE)/"pilot_eval.py"),"--dataset",str(DATA),"--preds","/kaggle/working/preds_F1_report.jsonl",
                    "--model","F1","--target-fpr",t,"--out",f"/kaggle/working/metrics_F1_{t}.json"],check=True)
print(open("/kaggle/working/runs/F1/run_summary.json").read())

In [ ]:
for t in ("0.009","0.015","0.02","0.03"):
    m=json.load(open(f"/kaggle/working/metrics_F1_{t}.json")); c=m["splits"]["val"]["combined"]; f=m["splits"]["val"]["fpr_per_negative_image"]
    print(t,"thr",m["thresholds"],"recall",round(c["recall"],4),"fpr",round(f["fpr"],4))
print("Independent predictions written: preds_F1_indep.jsonl")